# 08b · INT8 ablation analysis (no GPU)

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya Channarayapatna Ravishankar

**Priority:** 1 (follow-up to the pilot review; reads only)

**Purpose.** Evaluate every ablation engine under four threshold rules and compare the seed ranges of each setting with the natural setting.

**In plain words.** For each engine we read the saved dev and test video scores. Thresholds come only from the development set: A is the fixed FP32 threshold, B the engine's own EER point (kept for comparison), C the engine's own threshold that rejects 10% or 15% of real dev faces. A setting probably differs from natural when its range over the 3 seeds does not overlap natural's range.

**Expected inputs**
- Notebook 08 outputs: `results/int8/ablation/int8_entropy_<setting>_s<seed>/`; FP32 results `results/fp32/`; optionally the pilot results `results/int8/int8_entropy_natural_s<seed>/`.

**Expected outputs**
- `QPAD_Project/results/int8/ablation/analysis/` (`ablation_metrics.csv`, `ablation_ranges.csv`, `ablation_thresholds.csv`, `ablation_score_shifts.csv`, `pilot_vs_ablation_natural.csv`, and one figure per ablation line: `ablation_attack_share.png/.pdf`, `ablation_images.png/.pdf`)

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [12]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

qpad 0.1.0 from /content/drive/MyDrive/QPAD_Project/code/qpad


In [14]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

DATASET_ROOT       = /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw
  train    -> /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw/Train_files-002.tar
  dev      -> /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw/Dev_files-001.tar
  test     -> /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw/Test_files-003.tar
  metadata -> /content/drive/MyDrive/QPAD_Project/OULU_NPU/raw/OULU-NPU-20261010T032906Z-1-004.zip
PROJECT_DRIVE_ROOT = /content/drive/MyDrive/QPAD_Project
LOCAL_WORK_ROOT    = /content/qpad_work


In [15]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

{
 "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
 "packages": {
  "python": "3.13.18",
  "numpy": "2.4.6",
  "pandas": "2.2.3",
  "torch": "2.14.1+cpu",
  "torchvision": "0.29.1+cpu",
  "cv2": "5.0.0",
  "onnx": "1.23.2",
  "onnxruntime": "1.31.0",
  "tensorrt": null,
  "modelopt": null,
  "scipy": "1.16.3",
  "matplotlib": "3.10.0",
  "yaml": "6.0.3"
 },
 "gpu": {
  "cuda_available": false,
  "torch_cuda": null,
  "cudnn": null,
  "nvidia_smi": null
 }
}
git: {'commit': None, 'dirty': None}
Filesystem      Size  Used Avail Use% Mounted on
overlay         226G  261M  226G   1% /



In [16]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = False                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42

if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb08b", paths.logs / "nb08b.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

2026-11-12 09:12:14,143 INFO qpad.nb08b: DRY_RUN=False seed=42
Outputs of this run go to: /content/drive/MyDrive/QPAD_Project


In [17]:
import json, pandas as pd
from qpad import ablation as AB
from qpad.experiments import engine_tag
from qpad.pipeline import find_input
icfg, acfg = cfgs["int8"], cfgs["int8"]["ablation"]
METHOD = icfg["calibration_method"]["primary"]
TARGETS = [float(t) for t in acfg["bpcer_targets"]]
BASELINE = acfg["baseline"]
SETTINGS_TABLE = AB.settings_table(acfg, int(icfg["calibration_budget"]["frames_per_video"]),
                                   int(icfg["calibration_method"]["calibration_batch_size"]))
RESULT_DIRS, MISSING = {}, []
for setting in SETTINGS_TABLE["setting"]:
    for seed in acfg["seeds"]:
        d = find_input(paths, DRY_RUN, "results", f"int8/ablation/{engine_tag(METHOD, setting, seed)}")
        if (d / "metrics.json").is_file():
            RESULT_DIRS[(setting, seed)] = d
        else:
            MISSING.append(f"{setting} s{seed}")
print(f"engines with results: {len(RESULT_DIRS)}; missing: {MISSING or 'none'}")
if not RESULT_DIRS:
    raise FileNotFoundError("no ablation results found; run notebook 08 first")
FP32_DIR = find_input(paths, DRY_RUN, "results", "fp32")
ANALYSIS = OUT.results / "int8" / "ablation" / "analysis"
ANALYSIS.mkdir(parents=True, exist_ok=True)
print("writing to:", ANALYSIS)

engines with results: 24; missing: none
writing to: /content/drive/MyDrive/QPAD_Project/results/int8/ablation/analysis


## 1. Threshold rules
Each engine's thresholds under every rule, with the range over seeds. The pilot's EER thresholds (rule B) jumped between 0.51 and 0.76; the C rules use the 27th or 40th lowest real-face dev score instead of one borderline video, so they should move much less.

In [18]:
LONG = AB.collect(RESULT_DIRS, TARGETS)
LONG.to_csv(ANALYSIS / "ablation_metrics.csv", index=False)
fp32_thr = json.loads((FP32_DIR / "threshold.json").read_text())["threshold"]
FP32_RULES = AB.rule_metrics(AB.read_videos(FP32_DIR, "dev"), AB.read_videos(FP32_DIR, "test"),
                             {"A_fp32": fp32_thr, "B_dev_eer": fp32_thr}, TARGETS)
print("FP32 reference (same rules on the FP32 model):")
display(FP32_RULES[["rule", "threshold", "print_apcer_errors", "replay_apcer_errors", "bpcer_errors", "acer_pct"]])
THRESHOLDS = LONG.pivot_table(index=["setting", "seed"], columns="rule", values="threshold")
THRESHOLDS.to_csv(ANALYSIS / "ablation_thresholds.csv")
display(THRESHOLDS.round(4))
print("Spread of the threshold over all engines (max - min):", (THRESHOLDS.max() - THRESHOLDS.min()).round(4).to_dict())

FP32 reference (same rules on the FP32 model):


,rule,threshold,print_apcer_errors,replay_apcer_errors,bpcer_errors,acer_pct
0,A_fp32,0.731701,2,25,9,3.125000
1,B_dev_eer,0.731701,2,25,9,3.125000
2,C_bpcer10,0.989957,0,3,81,11.458333
3,C_bpcer15,0.995273,0,1,102,14.236111


rule            A_fp32  B_dev_eer  C_bpcer10  C_bpcer15
setting   seed                                         
attack0   1     0.7317     0.5152     0.9908     0.9943
          2     0.7317     0.5660     0.9914     0.9959
          3     0.7317     0.5056     0.9883     0.9940
attack10  1     0.7317     0.7148     0.9889     0.9944
          2     0.7317     0.7550     0.9944     0.9973
          3     0.7317     0.7074     0.9918     0.9953
attack20  1     0.7317     0.7443     0.9947     0.9981
          2     0.7317     0.7891     0.9971     0.9988
          3     0.7317     0.7218     0.9927     0.9956
attack40  1     0.7317     0.7575     0.9933     0.9970
          2     0.7317     0.7372     0.9950     0.9975
          3     0.7317     0.7143     0.9930     0.9955
images20  1     0.7317     0.5920     0.9867     0.9937
          2     0.7317     0.7576     0.9947     0.9973
          3     0.7317     0.7349     0.9951     0.9973
images200 1     0.7317     0.6719     0.9930     0.9954
          2     0.7317     0.7488     0.9956     0.9979
          3     0.7317     0.6954     0.9882     0.9938
images60  1     0.7317     0.7141     0.9859     0.9939
          2     0.7317     0.7630     0.9946     0.9968
          3     0.7317     0.7022     0.9767     0.9900
natural   1     0.7317     0.7385     0.9952     0.9973
          2     0.7317     0.7477     0.9955     0.9977
          3     0.7317     0.6716     0.9909     0.9941

Spread of the threshold over all engines (max - min): {'A_fp32': 0.0, 'B_dev_eer': 0.2835, 'C_bpcer10': 0.0204, 'C_bpcer15': 0.0087}


## 2. Seed ranges per setting
Min–max over seeds of the test error rates (%) under rules A, C10 and C15. `*` marks a range that does not overlap the natural setting's range. With 3 seeds this is a screening rule, not a significance test.

In [19]:
RANGES = AB.setting_ranges(LONG, BASELINE)
RANGES.to_csv(ANALYSIS / "ablation_ranges.csv", index=False)
MAIN_RULES = ["A_fp32", *[AB.bpcer_rule(t) for t in TARGETS]]
for rule in MAIN_RULES + ["B_dev_eer"]:
    print(AB.RULE_LABELS.get(rule, rule))
    display(AB.range_text(RANGES, [rule], ["print_apcer_pct", "replay_apcer_pct", "bpcer_pct"])
            .reindex(SETTINGS_TABLE["setting"]).dropna(how="all"))

A: fixed FP32 threshold


rule          A_fp32                                 
metric     bpcer_pct print_apcer_pct replay_apcer_pct
setting                                              
natural    1.67–3.06       0.28–1.39        5.56–9.17
attack40   1.67–2.78       1.11–1.11        6.39–9.44
attack20   1.39–2.78       0.56–1.67       6.67–10.28
attack10   1.94–2.78       0.56–1.11        5.83–8.06
attack0    2.22–3.06       0.56–0.56        5.56–6.67
images200  1.67–2.50       0.56–1.11        6.39–7.78
images60   2.22–3.06       0.28–1.11        5.00–8.89
images20   1.94–2.22       0.56–1.11        6.67–8.06

C: dev BPCER 10%


rule         C_bpcer10                                 
metric       bpcer_pct print_apcer_pct replay_apcer_pct
setting                                                
natural    23.06–25.28       0.00–0.00        0.28–0.83
attack40   21.94–24.72       0.00–0.00        0.28–0.83
attack20   21.94–23.89       0.00–0.00        0.56–0.83
attack10   22.22–23.33       0.00–0.00        0.56–0.56
attack0    21.94–23.89       0.00–0.00        0.56–0.83
images200  23.33–24.72       0.00–0.00        0.28–0.83
images60   20.56–24.44       0.00–0.00        0.56–0.83
images20   20.56–24.44       0.00–0.00        0.28–0.83

C: dev BPCER 15%


rule          C_bpcer15                                 
metric        bpcer_pct print_apcer_pct replay_apcer_pct
setting                                                 
natural     27.22–28.89       0.00–0.00        0.00–0.56
attack40    26.67–28.61       0.00–0.00        0.00–0.56
attack20   29.17–29.44*       0.00–0.00        0.00–0.28
attack10    28.33–30.00       0.00–0.00        0.28–0.28
attack0     26.94–29.44       0.00–0.00        0.00–0.00
images200   28.33–28.89       0.00–0.00        0.00–0.83
images60   29.17–29.72*       0.00–0.00        0.00–0.56
images20    27.22–30.00       0.00–0.00        0.00–0.28

B: engine dev EER


rule        B_dev_eer                                 
metric      bpcer_pct print_apcer_pct replay_apcer_pct
setting                                               
natural     1.67–2.50       1.11–1.39        7.50–9.17
attack40    1.67–2.22       0.83–1.11        6.67–8.61
attack20    1.67–2.50       0.56–1.11        6.67–8.06
attack10    2.22–2.22       0.56–1.39        6.39–7.50
attack0    0.28–0.83*      2.78–3.33*     11.39–12.78*
images200   1.39–2.22      0.56–0.83*        7.22–9.44
images60    2.22–3.06       0.28–1.11        5.56–7.50
images20    1.11–2.22       0.56–1.39       7.22–10.56

## 3. Figures: where do results start to get worse?

In [20]:
for line, levels in acfg["lines"].items():
    present = [s for s in levels if any(k[0] == s for k in RESULT_DIRS)]
    if present:
        print(AB.fig_ablation_line(LONG, SETTINGS_TABLE, line, present, MAIN_RULES, ANALYSIS, f"ablation_{line}")["png"])

/content/drive/MyDrive/QPAD_Project/results/int8/ablation/analysis/ablation_attack_share.png
/content/drive/MyDrive/QPAD_Project/results/int8/ablation/analysis/ablation_images.png


## 4. Score changes against natural (no threshold)
Mean change of the test video scores against the natural engine of the same seed. A positive mean on attacks means attacks look more like real faces. This does not depend on any threshold.

In [21]:
SHIFTS = AB.score_shifts_vs_baseline(RESULT_DIRS, BASELINE)
if len(SHIFTS):
    SHIFTS.to_csv(ANALYSIS / "ablation_score_shifts.csv", index=False)
    display(SHIFTS.groupby(["setting", "class"])[["mean_delta", "mean_abs_delta", "max_abs_delta"]]
            .agg(["min", "max"]).round(4).reindex(SETTINGS_TABLE["setting"], level=0))
else:
    print("only the baseline is available (dry run)")

mean_delta         mean_abs_delta         max_abs_delta  \
                           min     max            min     max           min   
setting   class                                                               
attack40  bona_fide    -0.0004  0.0034         0.0005  0.0034        0.0132   
          print        -0.0011  0.0106         0.0015  0.0107        0.0169   
          replay       -0.0039  0.0115         0.0040  0.0122        0.0160   
attack20  bona_fide     0.0034  0.0038         0.0034  0.0060        0.0732   
          print        -0.0008  0.0154         0.0117  0.0177        0.0634   
          replay       -0.0096  0.0287         0.0138  0.0288        0.0872   
attack10  bona_fide    -0.0041  0.0037         0.0029  0.0069        0.0926   
          print        -0.0214  0.0118         0.0111  0.0252        0.0699   
          replay       -0.0382  0.0144         0.0108  0.0410        0.0835   
attack0   bona_fide    -0.0048  0.0001         0.0027  0.0053        0.1391   
          print        -0.0197  0.0023         0.0050  0.0229        0.1155   
          replay       -0.0443 -0.0031         0.0083  0.0466        0.0932   
images200 bona_fide    -0.0034  0.0021         0.0011  0.0104        0.0287   
          print        -0.0179 -0.0025         0.0052  0.0269        0.0779   
          replay       -0.0215  0.0167         0.0057  0.0313        0.0581   
images60  bona_fide    -0.0089 -0.0033         0.0044  0.0107        0.1600   
          print        -0.0208 -0.0110         0.0202  0.0277        0.2239   
          replay       -0.0262  0.0047         0.0206  0.0360        0.1500   
images20  bona_fide    -0.0058  0.0114         0.0053  0.0116        0.1889   
          print        -0.0295  0.0247         0.0208  0.0319        0.2654   
          replay       -0.0320  0.0248         0.0218  0.0400        0.1572   

                             
                        max  
setting   class              
attack40  bona_fide  0.2419  
          print      0.1571  
          replay     0.2086  
attack20  bona_fide  0.2740  
          print      0.2013  
          replay     0.2173  
attack10  bona_fide  0.2749  
          print      0.2570  
          replay     0.2684  
attack0   bona_fide  0.1906  
          print      0.2405  
          replay     0.2584  
images200 bona_fide  0.1602  
          print      0.4347  
          replay     0.2653  
images60  bona_fide  0.2480  
          print      0.3862  
          replay     0.2946  
images20  bona_fide  0.3030  
          print      0.5056  
          replay     0.3018

## 5. Natural engines of this run against the pilot's
Natural seeds 1 and 2 use exactly the pilot's calibration images, so these pairs are rebuilds. Their difference is the noise floor that any ablation effect has to exceed. If the ablation ran on a different GPU than the pilot, the difference also includes the GPU change.

In [22]:
from qpad import diagnostics as D
PAIRS = []
for seed in acfg["seeds"]:
    tag = engine_tag(METHOD, BASELINE, seed)
    try:   # the pilot built only some seeds (natural s1, s2)
        pilot = find_input(paths, DRY_RUN, "results", f"int8/{tag}")
    except FileNotFoundError:
        continue
    if (pilot / "metrics.json").is_file() and (BASELINE, seed) in RESULT_DIRS:
        metas = [find_input(paths, DRY_RUN, "engines", t) / "engine_meta.json" for t in (tag, f"ablation/{tag}")]
        gpus = sorted({json.loads(m.read_text())["hardware"]["gpu_name"] for m in metas if m.is_file()})
        note = f" (different GPUs: {' vs '.join(gpus)})" if len(gpus) > 1 else ""
        PAIRS.append((f"pilot vs ablation natural s{seed}{note}", pilot, RESULT_DIRS[(BASELINE, seed)]))
if PAIRS:
    SHIFT_P, FLIPS_P = D.compare_pairs(PAIRS, "A")
    SHIFT_P.to_csv(ANALYSIS / "pilot_vs_ablation_natural.csv", index=False)
    display(SHIFT_P)
    display(FLIPS_P)
else:
    print("no pilot engine with the same seed found")

,comparison,class,mean_delta,mean_abs_delta,median_abs_delta,max_abs_delta,moved_over_0.01
0,pilot vs ablation natural s1 (different GPUs: ...,bona_fide,0.000791,0.000796,0.000014,0.018599,3
1,pilot vs ablation natural s1 (different GPUs: ...,print,0.002418,0.002458,0.000724,0.023554,19
2,pilot vs ablation natural s1 (different GPUs: ...,replay,0.004033,0.004040,0.002798,0.020685,37
3,pilot vs ablation natural s2 (different GPUs: ...,bona_fide,-0.000019,0.000147,0.000007,0.002980,0
4,pilot vs ablation natural s2 (different GPUs: ...,print,-0.000227,0.000667,0.000294,0.004603,0
5,pilot vs ablation natural s2 (different GPUs: ...,replay,-0.000384,0.000798,0.000568,0.005217,0


,comparison,policy,attack_rejected_to_accepted,attack_accepted_to_rejected,bona_fide_accepted_to_rejected,bona_fide_rejected_to_accepted,n_videos
0,pilot vs ablation natural s1 (different GPUs: ...,A,3,0,0,0,1080
1,pilot vs ablation natural s2 (different GPUs: ...,A,0,0,0,0,1080


In [23]:
CLEAR = RANGES[RANGES["separated"] & RANGES["rule"].isin(MAIN_RULES)
               & RANGES["metric"].isin(["print_apcer_pct", "replay_apcer_pct", "bpcer_pct"])]
print("Settings whose seed range is clear of natural's (rules A and C):")
print(CLEAR[["rule", "setting", "metric", "min", "max", "baseline_min", "baseline_max", "direction"]]
      .round(2).to_string(index=False) if len(CLEAR) else "none")

Settings whose seed range is clear of natural's (rules A and C):
     rule  setting    metric   min   max  baseline_min  baseline_max direction
C_bpcer15 attack20 bpcer_pct 29.17 29.44         27.22         28.89    higher
C_bpcer15 images60 bpcer_pct 29.17 29.72         27.22         28.89    higher


This is the last notebook of the ablation.